# Phân tích Thị trường Chứng khoán Đa cổ phiếu và Dự báo Xu hướng Giá sử dụng PySpark

## Tóm tắt Bài toán
Dự báo **xu hướng giá chứng khoán** (tăng/giảm) cho ngày tiếp theo dựa trên dữ liệu lịch sử từ **2013 đến 2026** sử dụng Machine Learning và PySpark.

**Mục tiêu chính:**
- Phân tích dữ liệu nhiều cổ phiếu (multi-stock)
- Xây dựng đặc trưng (features) hiệu quả bằng Window Functions
- Huấn luyện và so sánh 2 mô hình: Logistic Regression & Random Forest
- Đánh giá hiệu suất theo từng cổ phiếu
- Backtest chiến lược giao dịch dựa trên dự báo
- Tính lợi nhuận giả lập so với Buy & Hold strategy


## PHẦN 1: KHỞI TẠO PYSPARK VÀ IMPORT THƯ VIỆN

Trong phần này, chúng tôi sẽ:
- Import các thư viện cần thiết (PySpark, pandas, matplotlib, scikit-learn)
- Khởi tạo SparkSession cấu hình cho môi trường local
- Thiết lập cấu hình memory phù hợp

In [ ]:
# Import thư viện PySpark
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, LongType, TimestampType
from pyspark.sql.functions import (
    col, to_timestamp, lag, lead, avg, row_number, when, count, sum as spark_sum,
    min as spark_min, max as spark_max, round as spark_round, lit, concat_ws
)
from pyspark.sql.window import Window
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, VectorAssembler, StandardScaler
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator, BinaryClassificationEvaluator

# Import thư viện khác
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import os
import glob

# Cấu hình matplotlib
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("✓ Đã import tất cả thư viện thành công!")

ModuleNotFoundError: No module named 'numpy'

: 

In [ ]:
# Khởi tạo SparkSession cho local execution
spark = SparkSession.builder \
    .appName("StockAnalysisPySpark") \
    .master("local[*]") \
    .config("spark.sql.shuffle.partitions", "8") \
    .config("spark.driver.memory", "4g") \
    .config("spark.sql.adaptive.enabled", "true") \
    .getOrCreate()

# Cấu hình Spark
spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", "true")
spark.sparkContext.setLogLevel("ERROR")

print("✓ SparkSession đã khởi tạo thành công!")
print(f"  - Spark Version: {spark.version}")
print(f"  - Python Version: {pd.__version__}")
print(f"  - NumPy Version: {np.__version__}")

## PHẦN 2: NẠP VÀ KHÁM PHÁ DỮ LIỆU

Công việc trong phần này:
- Đọc tất cả file CSV từ thư mục `/home/admin/Documents/VSC/data/csv`
- Thêm cột `ticker` từ tên file
- Hiển thị schema, kiểu dữ liệu, và dòng dữ liệu mẫu
- Hiển thị thống kê cơ bản của các cột giá (open, high, low, close, volume)
- Liệt kê các cổ phiếu duy nhất (unique tickers)

In [ ]:
# Đường dẫn thư mục dữ liệu
data_dir = "/home/admin/Documents/VSC/data/csv"
csv_files = glob.glob(os.path.join(data_dir, "*.csv"))

print(f"📂 Tìm thấy {len(csv_files)} file CSV")
print(f"📍 Thư mục: {data_dir}\n")

# Đọc từng file và thêm cột ticker
dataframes = []
for csv_file in csv_files:
    ticker = os.path.basename(csv_file).replace('.csv', '')
    df_temp = spark.read.csv(csv_file, header=True, inferSchema=True)
    df_temp = df_temp.withColumn("ticker", lit(ticker))
    dataframes.append(df_temp)
    print(f"✓ Loaded: {ticker:6s} - {df_temp.count():,d} rows")

# Union tất cả dataframes
df_raw = dataframes[0]
for df_temp in dataframes[1:]:
    df_raw = df_raw.union(df_temp)

print(f"\n✓ Tổng dữ liệu: {df_raw.count():,d} rows từ {len(dataframes)} cổ phiếu")

In [ ]:
# Hiển thị schema và thông tin cơ bản
print("=" * 80)
print("SCHEMA CỦA DỮ LIỆU")
print("=" * 80)
df_raw.printSchema()

print("\n" + "=" * 80)
print("10 DÒNG DỮ LIỆU MẪU ĐẦU TIÊN")
print("=" * 80)
df_raw.show(10, truncate=False)

print("\n" + "=" * 80)
print("THỐNG KÊ CỌC BẢN")
print("=" * 80)
df_raw.describe(['open', 'high', 'low', 'close', 'volume']).show()

print("\n" + "=" * 80)
print("DANH SÁCH CỔ PHIẾU")
print("=" * 80)
tickers = df_raw.select("ticker").distinct().collect()
ticker_list = sorted([row.ticker for row in tickers])
print(f"Tổng {len(ticker_list)} cổ phiếu: {', '.join(ticker_list)}")

## PHẦN 3: TIỀN XỬ LÝ DỮ LIỆU (PYSPARK)

Công việc chính:
- **Convert kiểu dữ liệu**: Chuyển cột `time` → `timestamp`, các cột giá/khối lượng → `double`/`long`
- **Loại bỏ bản ghi trùng** và dòng lỗi (giá <= 0)
- **Xử lý missing values**: Drop hoặc fill tùy theo tính chất dữ liệu
- **Sort dữ liệu**: Theo ticker và date để chuẩn bị cho window functions
- **Kiểm tra chất lượng**: Xác minh dữ liệu sau xử lý

In [ ]:
# Step 1: Convert kiểu dữ liệu
print("Step 1: Convert kiểu dữ liệu...")
df_processed = df_raw \
    .withColumn("time", to_timestamp(col("time"), "yyyy-MM-dd")) \
    .withColumn("open", col("open").cast(DoubleType())) \
    .withColumn("high", col("high").cast(DoubleType())) \
    .withColumn("low", col("low").cast(DoubleType())) \
    .withColumn("close", col("close").cast(DoubleType())) \
    .withColumn("volume", col("volume").cast(LongType())) \
    .withColumn("ticker", col("ticker").cast(StringType()))

# Step 2: Loại bỏ null values
print("Step 2: Loại bỏ null values...")
null_count_before = df_processed.count()
df_processed = df_processed.dropna()
null_count_after = df_processed.count()
print(f"  - Loại bỏ {null_count_before - null_count_after:,d} dòng null")

# Step 3: Loại bỏ dòng lỗi (giá <= 0)
print("Step 3: Loại bỏ dòng lỗi (giá <= 0)...")
invalid_count = df_processed.filter((col("close") <= 0) | (col("volume") <= 0)).count()
df_processed = df_processed.filter((col("close") > 0) & (col("volume") > 0))
print(f"  - Loại bỏ {invalid_count:,d} dòng lỗi")

# Step 4: Loại bỏ bản ghi trùng
print("Step 4: Loại bỏ bản ghi trùng...")
df_processed = df_processed.dropDuplicates(['time', 'ticker'])

# Step 5: Sort theo ticker và date
print("Step 5: Sort theo ticker và date...")
df_processed = df_processed.sort(['ticker', 'time'])

print(f"\n✓ Dữ liệu sau xử lý: {df_processed.count():,d} rows")
print("\n10 dòng đầu sau xử lý:")
df_processed.show(10)

## PHẦN 4: PHÂN TÍCH DỮ LIỆU KHÁM PHÁ (EDA)

Mục đích:
- Chuyển Spark DataFrame sang pandas để vẽ biểu đồ
- Phân tích xu hướng giá theo thời gian cho các cổ phiếu đã chọn
- So sánh biến động giá giữa các cổ phiếu
- Phân tích mối quan hệ volume và giá
- Hiểu rõ đặc điểm dữ liệu trước khi xây dựng model

In [ ]:
# Chuyển dữ liệu Spark sang Pandas để vẽ biểu đồ
print("Chuyển dữ liệu Spark sang Pandas...")
df_pandas = df_processed.toPandas()
df_pandas['time'] = pd.to_datetime(df_pandas['time'])

print(f"✓ Chuyển xong: {len(df_pandas):,d} rows")

# Chọn một vài cổ phiếu đại diện để vẽ biểu đồ
selected_tickers = df_pandas['ticker'].unique()[:5]
print(f"Chọn {len(selected_tickers)} cổ phiếu đại diện: {list(selected_tickers)}")

# Vẽ biểu đồ giá theo thời gian
fig, axes = plt.subplots(len(selected_tickers), 1, figsize=(14, 12))
if len(selected_tickers) == 1:
    axes = [axes]

for idx, ticker in enumerate(selected_tickers):
    data = df_pandas[df_pandas['ticker'] == ticker].sort_values('time')
    axes[idx].plot(data['time'], data['close'], linewidth=1.5, color='steelblue', label='Close Price')
    axes[idx].fill_between(data['time'], data['low'], data['high'], alpha=0.3, color='lightblue', label='High-Low Range')
    axes[idx].set_title(f'{ticker} - Giá đóng cửa theo thời gian (2013-2026)', fontsize=12, fontweight='bold')
    axes[idx].set_ylabel('Giá (USD)', fontsize=10)
    axes[idx].legend(loc='upper left')
    axes[idx].grid(True, alpha=0.3)

plt.xlabel('Thời gian', fontsize=10)
plt.tight_layout()
plt.show()

print("✓ Biểu đồ vẽ xong!")

## PHẦN 5: FEATURE ENGINEERING 

Sử dụng **PySpark Window Functions** để tạo các đặc trưng dự báo:

### Các đặc trưng được tạo:

1. **Daily Return**: Tỷ suất lợi nhuận hàng ngày `(close - prev_close) / prev_close`
2. **Future Return**: Tỷ suất lợi nhuận ngày kế tiếp `(next_close - close) / close`
3. **Moving Averages**: MA5 (5 ngày), MA10 (10 ngày)
4. **Lag Features**: Giá ngày trước (lag1), 2 ngày trước (lag2), 3 ngày trước (lag3)
5. **Lag Return**: Return của ngày trước
6. **Target Label có ngưỡng**:
   - `1` nếu `future_return > +0.2%`
   - `0` nếu `future_return < -0.2%`
   - Các điểm nằm giữa 2 ngưỡng này được xem là nhiễu và sẽ loại bỏ
7. **Các feature mạnh hơn**:
   - RSI(14)
   - MACD và MACD Signal
   - Bollinger Bands (upper/lower/bandwidth)
   - Rolling volatility
   - Volume change
   - High-low range
   - Close-open return

Cách làm này giúp giảm nhiễu cho bài toán tăng/giảm, vì các phiên biến động rất nhỏ thường khó dự báo và làm model học kém ổn định.

Window function được phân chia theo **ticker** và sắp xếp theo **time** để đảm bảo tính chính xác.

## PHẦN 5A: XỬ LÝ VẤN ĐỀ VÀ LÝ DO THIẾT KẾ FEATURE

Trong bài toán dự báo xu hướng giá, 2 vấn đề chính thường làm mô hình học kém hiệu quả là:

1. **Nhiễu từ các phiên biến động nhỏ**
   - Những ngày tăng/giảm rất nhẹ thường không mang tín hiệu rõ ràng.
   - Nếu gán nhãn trực tiếp theo `next_close > close`, mô hình sẽ phải học cả các trường hợp mơ hồ, làm giảm độ chính xác.

2. **Thiếu thông tin ngữ cảnh thị trường**
   - Dữ liệu OHLCV thô chưa đủ để mô tả động lượng, xu hướng và độ biến động.
   - Vì vậy cần bổ sung các chỉ báo kỹ thuật như RSI, MACD, Bollinger Bands và các đặc trưng về volume, range, return.

### Cách xử lý đã áp dụng
- Dùng **label có ngưỡng**: chỉ coi là tăng/giảm khi `future_return` vượt quá ±0.2%.
- Loại bỏ các mẫu nằm giữa 2 ngưỡng để giảm nhiễu cho model.
- Bổ sung các feature mạnh hơn để mô hình có thêm thông tin:
  - RSI(14)
  - MACD và MACD Signal
  - Bollinger Bands
  - Rolling volatility
  - Volume change
  - High-low range
  - Close-open return

### Ý nghĩa cho báo cáo
Cách thiết kế này giúp mô hình tập trung vào các biến động có ý nghĩa hơn thay vì cố dự đoán mọi dao động nhỏ của thị trường. Đây là bước quan trọng để cải thiện khả năng tổng quát hóa và giảm sai số dự báo.

In [ ]:
# Định nghĩa Window Function: Phân chia theo ticker, sắp xếp theo time
windowSpec = Window.partitionBy('ticker').orderBy('time')

print("Bắt đầu Feature Engineering sử dụng Window Functions...")

# ── CẢI TIẾN 1: Tăng ngưỡng label lên 0.5% để giảm nhiễu ──
return_threshold = 0.005  # 0.5% (trước: 0.2%)
print(f"Ngưỡng label được đặt tại: {return_threshold*100:.2f}%")

# Feature 1: Lag(Close)
df_features = df_processed \
    .withColumn('lag1_close', lag('close', 1).over(windowSpec)) \
    .withColumn('lag2_close', lag('close', 2).over(windowSpec)) \
    .withColumn('lag3_close', lag('close', 3).over(windowSpec))

# Feature 2: Lead(Close) – dùng để tạo label
df_features = df_features \
    .withColumn('next_close', lead('close', 1).over(windowSpec))

# Feature 3: Future Return
df_features = df_features \
    .withColumn('future_return', (col('next_close') - col('close')) / col('close'))

# Feature 4: Daily Return
df_features = df_features \
    .withColumn('daily_return', (col('close') - col('lag1_close')) / col('lag1_close'))

# Feature 5: Lag Return
df_features = df_features \
    .withColumn('lag1_return', lag('daily_return', 1).over(windowSpec))

# Feature 6-7: MA5 / MA10
df_features = df_features \
    .withColumn('ma5',  avg('close').over(windowSpec.rowsBetween(-4,  0))) \
    .withColumn('ma10', avg('close').over(windowSpec.rowsBetween(-9,  0)))

# ── CẢI TIẾN 2: Thêm MA20 / MA50 ──
df_features = df_features \
    .withColumn('ma20', avg('close').over(windowSpec.rowsBetween(-19, 0))) \
    .withColumn('ma50', avg('close').over(windowSpec.rowsBetween(-49, 0)))

# ── CẢI TIẾN 2b: Price vs MA (momentum signal) ──
df_features = df_features \
    .withColumn('price_vs_ma5',  (col('close') - col('ma5'))  / col('ma5'))  \
    .withColumn('price_vs_ma20', (col('close') - col('ma20')) / col('ma20'))

# Feature 8: Rolling volatility 5-day
from pyspark.sql.functions import stddev_pop
df_features = df_features \
    .withColumn('rolling_volatility_5', stddev_pop('daily_return').over(windowSpec.rowsBetween(-4, 0)))

# Feature 9: RSI(14)
rsi_window = windowSpec.rowsBetween(-14, 0)
df_features = df_features \
    .withColumn('price_change', col('close') - col('lag1_close')) \
    .withColumn('gain', when(col('price_change') > 0, col('price_change')).otherwise(0.0)) \
    .withColumn('loss', when(col('price_change') < 0, -col('price_change')).otherwise(0.0)) \
    .withColumn('avg_gain_14', avg('gain').over(rsi_window)) \
    .withColumn('avg_loss_14', avg('loss').over(rsi_window)) \
    .withColumn('rs_14', when(col('avg_loss_14') == 0, None).otherwise(col('avg_gain_14') / col('avg_loss_14'))) \
    .withColumn('rsi_14', when(col('avg_loss_14') == 0, 100.0).otherwise(100 - (100 / (1 + col('rs_14')))))

# Feature 10: MACD proxy
df_features = df_features \
    .withColumn('ema12_proxy', avg('close').over(windowSpec.rowsBetween(-11, 0))) \
    .withColumn('ema26_proxy', avg('close').over(windowSpec.rowsBetween(-25, 0))) \
    .withColumn('macd', col('ema12_proxy') - col('ema26_proxy')) \
    .withColumn('macd_signal', avg('macd').over(windowSpec.rowsBetween(-8, 0)))

# Feature 11: Bollinger Bands (20-day)
df_features = df_features \
    .withColumn('bb_mid', avg('close').over(windowSpec.rowsBetween(-19, 0))) \
    .withColumn('bb_std', stddev_pop('close').over(windowSpec.rowsBetween(-19, 0))) \
    .withColumn('bb_upper', col('bb_mid') + (2 * col('bb_std'))) \
    .withColumn('bb_lower', col('bb_mid') - (2 * col('bb_std'))) \
    .withColumn('bb_bandwidth', when(col('bb_mid') != 0, (col('bb_upper') - col('bb_lower')) / col('bb_mid')).otherwise(None))

# Feature 12: Volume-based
df_features = df_features \
    .withColumn('lag1_volume', lag('volume', 1).over(windowSpec)) \
    .withColumn('volume_change', when(col('lag1_volume').isNull() | (col('lag1_volume') == 0), None)
                .otherwise((col('volume') - col('lag1_volume')) / col('lag1_volume'))) \
    .withColumn('high_low_range', when(col('close') != 0, (col('high') - col('low')) / col('close')).otherwise(None)) \
    .withColumn('close_open_return', when(col('open') != 0, (col('close') - col('open')) / col('open')).otherwise(None))

# ── CẢI TIẾN 3: Stochastic %K(14) ──
stoch_w = windowSpec.rowsBetween(-13, 0)
df_features = df_features \
    .withColumn('low14',  spark_min('low').over(stoch_w)) \
    .withColumn('high14', spark_max('high').over(stoch_w)) \
    .withColumn('stoch_k',
        when(col('high14') == col('low14'), 50.0)
        .otherwise((col('close') - col('low14')) / (col('high14') - col('low14')) * 100))

# ── CẢI TIẾN 4: ATR(14) – Average True Range ──
df_features = df_features \
    .withColumn('atr14', avg(col('high') - col('low')).over(windowSpec.rowsBetween(-13, 0)))

# ── CẢI TIẾN 5: OBV direction signal ──
df_features = df_features \
    .withColumn('price_dir',
        when(col('close') > col('lag1_close'), 1.0)
        .when(col('close') < col('lag1_close'), -1.0)
        .otherwise(0.0)) \
    .withColumn('obv_signal', avg('price_dir').over(windowSpec.rowsBetween(-4, 0)))

# Feature 13: Target Label với ngưỡng 0.5%
df_features = df_features \
    .withColumn(
        'label',
        when(col('future_return') > return_threshold, 1)
        .when(col('future_return') < -return_threshold, 0)
        .otherwise(None)
    )

print("\u2713 Feature Engineering xong!")
print(f"\nDữ liệu sau feature engineering: {df_features.count():,d} rows")
df_features.select('time', 'ticker', 'close', 'future_return',
                   'rsi_14', 'macd', 'stoch_k', 'atr14', 'obv_signal', 'label').show(5)


## PHẦN 6: LÀM SẠCH DỮ LIỆU SAU FEATURE ENGINEERING

Sau khi tạo features bằng lag() và window functions, sẽ có dòng null:
- Dòng đầu tiên mỗi ticker sẽ có null trong lag columns
- Dòng cuối cùng mỗi ticker sẽ có null trong `next_close` và `future_return`
- Các dòng có biến động nhỏ quanh 0 cũng được gán `label = null` để loại bỏ nhiễu

Bước này sẽ:
- **Drop tất cả null rows**
- **Xác minh dữ liệu đã sạch**
- **Kiểm tra phân phối label** (có cân bằng không?)

In [ ]:
# Drop null values do lag/window functions
print("Dropping null values...")
count_before = df_features.count()
df_features = df_features.dropna()
count_after = df_features.count()
null_removed = count_before - count_after

print(f"✓ Loại bỏ {null_removed:,d} rows null ({null_removed/count_before*100:.1f}%)")
print(f"  Dữ liệu sau drop null: {count_after:,d} rows\n")

# Kiểm tra phân phối label
print("=" * 60)
print("PHÂN PHỐI LABEL")
print("=" * 60)
label_dist = df_features.groupBy('label').count().sort('label').collect()
for row in label_dist:
    percentage = (row['count'] / count_after) * 100
    print(f"  Label {int(row['label'])}: {row['count']:,d} ({percentage:.1f}%)")

# Kiểm tra phân phối label theo ticker
print("\n" + "=" * 60)
print("PHÂN PHỐI LABEL THEO TICKER")
print("=" * 60)
label_by_ticker = (
    df_features
    .groupBy("ticker")
    .pivot("label", [0, 1])
    .count()
    .na.fill(0)
    .select(
        "ticker",
        col("0").alias("label_0_count"),
        col("1").alias("label_1_count")
    )
    .orderBy("ticker")
)
label_by_ticker.show()

# Số dòng dữ liệu được sử dụng cho model
print(f"\n✓ Tổng dữ liệu có thể sử dụng cho model: {count_after:,d} rows")

## PHẦN 7: CHIA DỮ LIỆU THEO THỜI GIAN (TIME SERIES SPLIT)

Để tránh **data leakage** trong time series, ta không dùng **random split**. Thay vào đó:

- **Training Set**: 2013-01-01 → 2021-12-31 (dùng để huấn luyện model)
- **Testing Set**: 2022-01-01 → 2026-12-31 (dùng để đánh giá model)

Phương pháp này phù hợp với thực tế: dự báo tương lai dựa trên quá khứ, không phải ngược lại.

In [ ]:
# Chia dữ liệu theo thời gian (Time Series Split)
from pyspark.sql.functions import year

# Tạo cột year để dễ filter
df_features = df_features.withColumn('year', year('time'))

# Training data: 2013-2021
df_train = df_features.filter(col('year') <= 2021)

# Testing data: 2022-2026
df_test = df_features.filter(col('year') >= 2022)

# Thống kê
train_count = df_train.count()
test_count = df_test.count()
total_count = df_features.count()

print("=" * 80)
print("CHIA DỮ LIỆU THEO THỜI GIAN")
print("=" * 80)
print(f"\n📊 Training Set (2013-2021): {train_count:,d} rows ({train_count/total_count*100:.1f}%)")
print(f"📊 Testing Set (2022-2026):  {test_count:,d} rows ({test_count/total_count*100:.1f}%)")
print(f"📊 Tổng cộng:                {total_count:,d} rows\n")

# Thống kê theo ticker
print("Training set by ticker:")
df_train.groupBy('ticker').count().sort('ticker').show(truncate=False)

print("\nTesting set by ticker:")
df_test.groupBy('ticker').count().sort('ticker').show(truncate=False)

## PHẦN 8: CHUẨN BỊ DỮ LIỆU CHO ML MODELS

Bước này xử lý:
- **StringIndexer**: Encode cột `ticker` (text) thành số
- **VectorAssembler**: Gộp tất cả features thành một cột vector
- **StandardScaler** (nếu cần): Chuẩn hóa giá trị features về cùng scale

Feature columns được sử dụng:
- `daily_return`, `lag1_return`, `lag1_close`, `lag2_close`, `lag3_close`
- `ma5`, `ma10`, `rolling_volatility_5`, `rsi_14`, `macd`, `macd_signal`
- `bb_bandwidth`, `volume_change`, `high_low_range`, `close_open_return`
- `lag1_volume`, `ticker_idx` (encoded ticker)
- `volume`

In [ ]:
# Step 1: StringIndexer – Encode ticker
print("Step 1: Encode ticker với StringIndexer...")
ticker_indexer = StringIndexer(inputCol="ticker", outputCol="ticker_idx", handleInvalid="keep")
ticker_indexer_model = ticker_indexer.fit(df_train)
df_train_indexed = ticker_indexer_model.transform(df_train)
df_test_indexed  = ticker_indexer_model.transform(df_test)

# ── CẢI TIẾN: Bổ sung features mới vào danh sách ──
feature_columns = [
    'daily_return', 'lag1_return', 'lag1_close', 'lag2_close', 'lag3_close',
    'ma5', 'ma10', 'ma20', 'ma50',
    'price_vs_ma5', 'price_vs_ma20',
    'rolling_volatility_5', 'rsi_14', 'macd', 'macd_signal',
    'bb_bandwidth', 'volume_change', 'high_low_range', 'close_open_return',
    'lag1_volume', 'ticker_idx', 'volume',
    'stoch_k', 'atr14', 'obv_signal'
]
print(f"Số features: {len(feature_columns)}")

# Kiểm tra missing
print("\nKiểm tra missing values trong features:")
missing_counts = df_train_indexed.select(
    *[count(when(col(c).isNull(), 1)).alias(c) for c in feature_columns]
).collect()[0]
for c in feature_columns:
    m = missing_counts[c]
    status = f"  \u26a0\ufe0f {c}: {m:,d} null" if m > 0 else f"  \u2713 {c}: OK"
    print(status)

# Step 2: VectorAssembler
print("\nStep 2: Gộp features với VectorAssembler...")
assembler = VectorAssembler(inputCols=feature_columns, outputCol="raw_features", handleInvalid="skip")
df_train_assembled = assembler.transform(df_train_indexed)
df_test_assembled  = assembler.transform(df_test_indexed)

# ── CẢI TIẾN: StandardScaler (quan trọng cho Logistic Regression) ──
print("Step 3: StandardScaler...")
scaler = StandardScaler(inputCol="raw_features", outputCol="features",
                        withMean=True, withStd=True)
scaler_model = scaler.fit(df_train_assembled)
df_train_assembled = scaler_model.transform(df_train_assembled)
df_test_assembled  = scaler_model.transform(df_test_assembled)

# ── CẢI TIẾN: Class Weight để cân bằng imbalance ──
print("Step 4: Tính class weights...")
label_counts = {r['label']: r['count'] for r in df_train.groupBy('label').count().collect()}
total_rows = sum(label_counts.values())
w0 = total_rows / (2.0 * label_counts.get(0.0, 1))
w1 = total_rows / (2.0 * label_counts.get(1.0, 1))
print(f"  Label 0 (giảm): {label_counts.get(0.0, 0):,d} rows → weight = {w0:.4f}")
print(f"  Label 1 (tăng): {label_counts.get(1.0, 0):,d} rows → weight = {w1:.4f}")

df_train_assembled = df_train_assembled.withColumn(
    'weight',
    when(col('label') == 1.0, w1).otherwise(w0)
)
df_test_assembled = df_test_assembled.withColumn('weight', when(col('label') == 1.0, w1).otherwise(w0))

print(f"\n\u2713 Training data: {df_train_assembled.count():,d} rows")
print(f"\u2713 Testing data:  {df_test_assembled.count():,d} rows")
df_train_assembled.select('time', 'ticker', 'close', 'label', 'weight', 'features').show(3)


## PHẦN 9: HUẤN LUYỆN MÔ HÌNH LOGISTIC REGRESSION

**Logistic Regression** là một mô hình tuyến tính cổ điển cho bài toán phân loại nhị phân (binary classification).

Ưu điểm:
- Nhanh, dễ hiểu
- Cho xác suất dự báo
- Ít overfitting

Tham số:
- `maxIter`: Số vòng lặp tối đa (gradient descent)
- `regParam`: Tham số regularization (L2)
- `elasticNetParam`: Cân bằng L1/L2

In [ ]:
# Logistic Regression – cải tiến với scaled features và tuned hyperparams
print("=" * 80)
print("HUẤN LUYỆN LOGISTIC REGRESSION (IMPROVED)")
print("=" * 80)

lr = LogisticRegression(
    featuresCol="features",
    labelCol="label",
    weightCol="weight",
    maxIter=200,        # tăng từ 100 → 200
    regParam=0.001,     # giảm regularization
    elasticNetParam=0.0
)

print("\nHuấn luyện trên training data...")
lr_model = lr.fit(df_train_assembled)
print("\u2713 Huấn luyện xong!")

print("\nDự báo trên test set...")
lr_predictions = lr_model.transform(df_test_assembled)
print("\u2713 Dự báo xong!")

lr_predictions.select('time', 'ticker', 'close', 'label', 'prediction', 'probability').show(10)
print(f"\n\u2713 Logistic Regression training xong! Intercept: {lr_model.intercept:.4f}")


## PHẦN 10: HUẤN LUYỆN MÔ HÌNH RANDOM FOREST

**Random Forest** là một ensemble method dùng nhiều decision trees để dự báo.

Ưu điểm:
- Có thể capture non-linear relationships
- Đánh giá được feature importance
- Ít bị overfitting hơn single tree (do ensemble)

Tham số:
- `numTrees`: Số cây quyết định
- `maxDepth`: Độ sâu tối đa của mỗi cây
- `minInstancesPerNode`: Số sample tối thiểu tại mỗi node lá

In [ ]:
# Random Forest – cải tiến với numTrees=150, maxDepth=12, weightCol
print("=" * 80)
print("HUẤN LUYỆN RANDOM FOREST (IMPROVED)")
print("=" * 80)

rf = RandomForestClassifier(
    featuresCol="features",
    labelCol="label",
    weightCol="weight",
    numTrees=150,              # tăng từ 50 → 150
    maxDepth=12,               # tăng từ 10 → 12
    minInstancesPerNode=3,     # giảm từ 5 → 3
    featureSubsetStrategy="sqrt",
    seed=42
)

print("\nHuấn luyện trên training data...")
rf_model = rf.fit(df_train_assembled)
print("\u2713 Huấn luyện xong!")

print("\nDự báo trên test set...")
rf_predictions = rf_model.transform(df_test_assembled)
print("\u2713 Dự báo xong!")

# Feature Importance
print("\n" + "=" * 80)
print("FEATURE IMPORTANCE (Random Forest)")
print("=" * 80)
feature_importance = sorted(
    zip(feature_columns, rf_model.featureImportances),
    key=lambda x: x[1], reverse=True
)
for feat, imp in feature_importance:
    bar = "█" * int(imp * 200)
    print(f"  {feat:25s}: {imp:.4f}  {bar}")

rf_predictions.select('time', 'ticker', 'close', 'label', 'prediction', 'probability').show(10)


## PHẦN 11: ĐÁNH GIÁ MÔ HÌNH VÀ SO SÁNH

Các metric đánh giá:
- **Accuracy**: Tỷ lệ dự báo đúng = (TP + TN) / (TP + TN + FP + FN)
- **Precision**: Trong những dự báo "tăng", bao nhiêu % là đúng = TP / (TP + FP)
- **Recall**: Trong những trường hợp thực tế "tăng", bao nhiêu % được dự báo đúng = TP / (TP + FN)
- **F1-Score**: Trung bình hài hòa của Precision và Recall

So sánh: Logistic Regression vs Random Forest

In [ ]:
# Đánh giá Logistic Regression
print("=" * 80)
print("ĐÁNH GIÁ MÔ HÌNH - LOGISTIC REGRESSION")
print("=" * 80)

evaluator = MulticlassClassificationEvaluator(
    predictionCol="prediction",
    labelCol="label",
    metricName="accuracy"
)

lr_accuracy = evaluator.evaluate(lr_predictions)
print(f"\n🎯 Accuracy (LR): {lr_accuracy:.4f} ({lr_accuracy*100:.2f}%)")

# Confusion Matrix - Logistic Regression
print("\nConfusion Matrix - Logistic Regression:")
lr_cm = lr_predictions.groupBy("label", "prediction").count()
lr_cm.show()

# Đánh giá Random Forest
print("\n" + "=" * 80)
print("ĐÁNH GIÁ MÔ HÌNH - RANDOM FOREST")
print("=" * 80)

rf_accuracy = evaluator.evaluate(rf_predictions)
print(f"\n🎯 Accuracy (RF): {rf_accuracy:.4f} ({rf_accuracy*100:.2f}%)")

# Confusion Matrix - Random Forest
print("\nConfusion Matrix - Random Forest:")
rf_cm = rf_predictions.groupBy("label", "prediction").count()
rf_cm.show()

# So sánh 2 model
print("\n" + "=" * 80)
print("SO SÁNH 2 MÔ HÌNH")
print("=" * 80)
print(f"\n📊 Logistic Regression Accuracy: {lr_accuracy:.4f} ({lr_accuracy*100:.2f}%)")
print(f"📊 Random Forest Accuracy:       {rf_accuracy:.4f} ({rf_accuracy*100:.2f}%)")
print(f"📊 Chênh lệch (RF - LR):         {(rf_accuracy - lr_accuracy):.4f}")

if rf_accuracy > lr_accuracy:
    print(f"\n✅ Random Forest tốt hơn Logistic Regression ({(rf_accuracy - lr_accuracy)*100:.2f}%)")
elif lr_accuracy > rf_accuracy:
    print(f"\n✅ Logistic Regression tốt hơn Random Forest ({(lr_accuracy - rf_accuracy)*100:.2f}%)")
else:
    print(f"\n⚖️ Hai mô hình có hiệu suất tương đương")

## PHẦN 11B: ĐÁNH GIÁ BẰNG SAI SỐ DỰ BÁO VÀ THỬ MÔ HÌNH NÂNG CAO

Ngoài Accuracy, phần này bổ sung các thước đo sai số để nhìn rõ hơn chất lượng dự báo:
- **Error rate** = 1 - Accuracy
- **Log Loss**: phạt mạnh các dự báo xác suất sai tự tin
- **Brier Score**: đo sai số bình phương giữa xác suất dự báo và nhãn thực

Sau đó thử thêm **GBTClassifier** để so sánh với Logistic Regression và Random Forest.

In [ ]:
# GBTClassifier – cải tiến với maxIter=150, maxDepth=7, stepSize=0.05
print("=" * 80)
print("HUẤN LUYỆN GBTCLASSIFIER (IMPROVED)")
print("=" * 80)

from pyspark.ml.classification import GBTClassifier

gbt = GBTClassifier(
    featuresCol='features',
    labelCol='label',
    maxIter=150,          # tăng từ 50 → 150
    maxDepth=7,           # tăng từ 5 → 7
    stepSize=0.05,        # giảm từ 0.1 → 0.05 (ít overfit hơn)
    subsamplingRate=0.8,  # thêm mới: subsampling giảm variance
    seed=42
)

gbt_model = gbt.fit(df_train_assembled)
gbt_predictions = gbt_model.transform(df_test_assembled)

evaluator = MulticlassClassificationEvaluator(
    labelCol='label', predictionCol='prediction', metricName='accuracy'
)
logloss_evaluator = MulticlassClassificationEvaluator(
    labelCol='label', probabilityCol='probability', metricName='logLoss'
)

lr_accuracy  = evaluator.evaluate(lr_predictions)
rf_accuracy  = evaluator.evaluate(rf_predictions)
gbt_accuracy = evaluator.evaluate(gbt_predictions)

lr_error_rate  = 1 - lr_accuracy
rf_error_rate  = 1 - rf_accuracy
gbt_error_rate = 1 - gbt_accuracy

lr_logloss  = logloss_evaluator.evaluate(lr_predictions)
rf_logloss  = logloss_evaluator.evaluate(rf_predictions)
gbt_logloss = logloss_evaluator.evaluate(gbt_predictions)

lr_prob_pd  = lr_predictions.select('label','probability').toPandas()
rf_prob_pd  = rf_predictions.select('label','probability').toPandas()
gbt_prob_pd = gbt_predictions.select('label','probability').toPandas()

lr_prob_pd['p1']  = lr_prob_pd['probability'].apply(lambda v: float(v[1]))
rf_prob_pd['p1']  = rf_prob_pd['probability'].apply(lambda v: float(v[1]))
gbt_prob_pd['p1'] = gbt_prob_pd['probability'].apply(lambda v: float(v[1]))

lr_brier  = np.mean((lr_prob_pd['label']  - lr_prob_pd['p1'])  ** 2)
rf_brier  = np.mean((rf_prob_pd['label']  - rf_prob_pd['p1'])  ** 2)
gbt_brier = np.mean((gbt_prob_pd['label'] - gbt_prob_pd['p1']) ** 2)

compare_df = pd.DataFrame([
    {'Model': 'Logistic Regression', 'Accuracy': lr_accuracy,  'ErrorRate': lr_error_rate,  'LogLoss': lr_logloss,  'BrierScore': lr_brier},
    {'Model': 'Random Forest',       'Accuracy': rf_accuracy,  'ErrorRate': rf_error_rate,  'LogLoss': rf_logloss,  'BrierScore': rf_brier},
    {'Model': 'GBTClassifier',       'Accuracy': gbt_accuracy, 'ErrorRate': gbt_error_rate, 'LogLoss': gbt_logloss, 'BrierScore': gbt_brier},
]).sort_values('Accuracy', ascending=False)

print("\n📊 KẾT QUẢ SO SÁNH 3 MÔ HÌNH (IMPROVED):")
print(compare_df.to_string(index=False))

best_model_name = compare_df.iloc[0]['Model']
best_accuracy   = compare_df.iloc[0]['Accuracy']
print(f"\n\U0001f3c6 Mô hình tốt nhất: {best_model_name} ({best_accuracy*100:.2f}%)")


## PHẦN 11D: CROSS VALIDATION – TỐI ƯU SIÊU THAM SỐ

Dùng **3-Fold CrossValidator** để tìm bộ tham số tốt nhất cho Random Forest.

- Grid search trên `numTrees`, `maxDepth`, `minInstancesPerNode`
- Kết quả sẽ cho thấy sự cải thiện so với baseline RF

In [ ]:
# CrossValidator – Tìm siêu tham số tốt nhất cho Random Forest
print("=" * 80)
print("CROSS VALIDATION – RANDOM FOREST GRID SEARCH")
print("=" * 80)

from pyspark.ml.tuning import CrossValidator, ParamGridBuilder

rf_cv_base = RandomForestClassifier(
    featuresCol='features', labelCol='label',
    weightCol='weight', seed=42
)

param_grid = (ParamGridBuilder()
    .addGrid(rf_cv_base.numTrees,             [100, 150, 200])
    .addGrid(rf_cv_base.maxDepth,             [10, 12])
    .addGrid(rf_cv_base.minInstancesPerNode,  [2, 3])
    .build())

cv_evaluator = MulticlassClassificationEvaluator(
    labelCol='label', predictionCol='prediction', metricName='accuracy'
)

cv = CrossValidator(
    estimator=rf_cv_base,
    estimatorParamMaps=param_grid,
    evaluator=cv_evaluator,
    numFolds=3,
    seed=42
)

print(f"Đang chạy {len(param_grid)} tổ hợp tham số x 3 folds = {len(param_grid)*3} lần train...")
print("(Có thể mất vài phút...)")
cv_model = cv.fit(df_train_assembled)
cv_predictions = cv_model.transform(df_test_assembled)
cv_accuracy = cv_evaluator.evaluate(cv_predictions)

best_rf_cv = cv_model.bestModel
print(f"\n\u2713 CrossValidator xong!")
print(f"  Best numTrees:            {best_rf_cv.getNumTrees}")
print(f"  Best maxDepth:            {best_rf_cv.getOrDefault('maxDepth')}")
print(f"  Best minInstancesPerNode: {best_rf_cv.getOrDefault('minInstancesPerNode')}")
print(f"  CV Best Accuracy:         {cv_accuracy:.4f} ({cv_accuracy*100:.2f}%)")
print(f"  Baseline RF Accuracy:     {rf_accuracy:.4f} ({rf_accuracy*100:.2f}%)")
print(f"  Improvement:              +{(cv_accuracy - rf_accuracy)*100:.2f} điểm %")


In [ ]:
# Thử nghiệm 1: Quét ngưỡng label và so sánh accuracy
from pyspark.sql.functions import year

experiment_feature_cols = [
    'daily_return', 'lag1_return', 'lag1_close', 'lag2_close', 'lag3_close',
    'ma5', 'ma10', 'rolling_volatility_5', 'rsi_14', 'macd', 'macd_signal',
    'bb_bandwidth', 'volume_change', 'high_low_range', 'close_open_return',
    'lag1_volume', 'volume'
]

# Dùng lại bảng feature đã có, chỉ gán lại label theo nhiều ngưỡng
threshold_grid = [0.001, 0.002, 0.003, 0.005]  # 0.1%, 0.2%, 0.3%, 0.5%
threshold_results = []

for th in threshold_grid:
    df_exp = (
        df_features
        .withColumn('label_exp',
            when(col('future_return') > th, 1)
            .when(col('future_return') < -th, 0)
            .otherwise(None)
        )
        .dropna(subset=experiment_feature_cols + ['label_exp', 'time', 'ticker'])
        .withColumn('year_exp', year('time'))
    )

    df_train_exp = df_exp.filter(col('year_exp') <= 2021)
    df_test_exp = df_exp.filter(col('year_exp') >= 2022)

    # Encode ticker dựa trên train để tránh leakage
    idx_exp = StringIndexer(inputCol='ticker', outputCol='ticker_idx_exp', handleInvalid='keep')
    idx_model = idx_exp.fit(df_train_exp)
    train_idx = idx_model.transform(df_train_exp)
    test_idx = idx_model.transform(df_test_exp)

    asm_exp = VectorAssembler(
        inputCols=experiment_feature_cols + ['ticker_idx_exp'],
        outputCol='features_exp',
        handleInvalid='skip'
    )
    train_asm = asm_exp.transform(train_idx)
    test_asm = asm_exp.transform(test_idx)

    rf_exp = RandomForestClassifier(
        featuresCol='features_exp',
        labelCol='label_exp',
        numTrees=80,
        maxDepth=10,
        minInstancesPerNode=5,
        seed=42
    )

    model_exp = rf_exp.fit(train_asm)
    pred_exp = model_exp.transform(test_asm)

    eval_exp = MulticlassClassificationEvaluator(
        predictionCol='prediction',
        labelCol='label_exp',
        metricName='accuracy'
    )
    acc_exp = eval_exp.evaluate(pred_exp)

    threshold_results.append({
        'threshold': th,
        'train_rows': train_asm.count(),
        'test_rows': test_asm.count(),
        'accuracy': acc_exp,
        'error_rate': 1 - acc_exp
    })

threshold_df = pd.DataFrame(threshold_results).sort_values('accuracy', ascending=False)
print('Kết quả quét ngưỡng label:')
print(threshold_df.to_string(index=False))

best_threshold = float(threshold_df.iloc[0]['threshold'])
print(f"\nNgưỡng tốt nhất theo accuracy: {best_threshold*100:.2f}%")

In [ ]:
# Thử nghiệm 2: Tune Random Forest tại ngưỡng tốt nhất
from pyspark.ml.tuning import ParamGridBuilder, TrainValidationSplit

# Chuẩn bị lại data theo best threshold
df_best = (
    df_features
    .withColumn('label_best',
        when(col('future_return') > best_threshold, 1)
        .when(col('future_return') < -best_threshold, 0)
        .otherwise(None)
    )
    .dropna(subset=experiment_feature_cols + ['label_best', 'time', 'ticker'])
    .withColumn('year_best', year('time'))
)

train_best = df_best.filter(col('year_best') <= 2021)
test_best = df_best.filter(col('year_best') >= 2022)

idx_best = StringIndexer(inputCol='ticker', outputCol='ticker_idx_best', handleInvalid='keep')
idx_best_model = idx_best.fit(train_best)
train_best_idx = idx_best_model.transform(train_best)
test_best_idx = idx_best_model.transform(test_best)

asm_best = VectorAssembler(
    inputCols=experiment_feature_cols + ['ticker_idx_best'],
    outputCol='features_best',
    handleInvalid='skip'
)

train_best_asm = asm_best.transform(train_best_idx)
test_best_asm = asm_best.transform(test_best_idx)

rf_base = RandomForestClassifier(
    featuresCol='features_best',
    labelCol='label_best',
    seed=42
)

param_grid = (ParamGridBuilder()
    .addGrid(rf_base.numTrees, [80, 120, 200])
    .addGrid(rf_base.maxDepth, [8, 10, 12])
    .addGrid(rf_base.minInstancesPerNode, [2, 5])
    .build())

evaluator_best = MulticlassClassificationEvaluator(
    predictionCol='prediction',
    labelCol='label_best',
    metricName='accuracy'
)

tvs = TrainValidationSplit(
    estimator=rf_base,
    estimatorParamMaps=param_grid,
    evaluator=evaluator_best,
    trainRatio=0.8,
    seed=42
)

rf_tuned_model = tvs.fit(train_best_asm)
rf_tuned_pred = rf_tuned_model.transform(test_best_asm)
rf_tuned_accuracy = evaluator_best.evaluate(rf_tuned_pred)

best_rf = rf_tuned_model.bestModel
print('Kết quả tune Random Forest:')
print(f"  - Best numTrees: {best_rf.getNumTrees}")
print(f"  - Best maxDepth: {best_rf.getOrDefault('maxDepth')}")
print(f"  - Best minInstancesPerNode: {best_rf.getOrDefault('minInstancesPerNode')}")
print(f"  - Tuned Accuracy: {rf_tuned_accuracy:.4f} ({rf_tuned_accuracy*100:.2f}%)")

# So sánh trước/sau
baseline_rf = rf_accuracy if 'rf_accuracy' in globals() else None
if baseline_rf is not None:
    print(f"  - Baseline RF Accuracy: {baseline_rf:.4f} ({baseline_rf*100:.2f}%)")
    print(f"  - Improvement: {(rf_tuned_accuracy - baseline_rf):.4f} ({(rf_tuned_accuracy - baseline_rf)*100:.2f} điểm)")

## PHẦN 11C: THỬ NGHIỆM TĂNG ĐỘ CHÍNH XÁC (THRESHOLD + TUNING)

Phần này thử 2 hướng cải thiện trực tiếp:

1. Quét nhiều ngưỡng label (`future_return`) để tìm ngưỡng cho tín hiệu tốt nhất.
2. Tune hyperparameter cho Random Forest tại ngưỡng tốt nhất.

Mục tiêu: tối ưu Accuracy nhưng vẫn theo đúng time-series split.

## PHẦN 12: PHÂN TÍCH HIỆU NĂNG THEO TỪNG CỔ PHIẾU

Mục đích:
- Tính accuracy riêng cho từng ticker
- Xác định cổ phiếu nào dự báo tốt, cổ phiếu nào kém
- Hiểu các đặc điểm thị trường của từng mã chứng khoán

Câu hỏi:
- Model nào dự báo tốt nhất cho mỗi ticker?
- Có ticker nào mà cả 2 model đều cho kết quả kém?

In [ ]:
# Phân tích accuracy theo ticker - Logistic Regression
print("=" * 100)
print("ACCURACY THEO TICKER - LOGISTIC REGRESSION")
print("=" * 100)

lr_by_ticker = lr_predictions.withColumn('correct', col('prediction') == col('label')) \
    .groupBy('ticker') \
    .agg(
        (spark_sum(col('correct').cast('int')) / count('*')).alias('accuracy'),
        count('*').alias('count')
    ) \
    .sort(col('accuracy').desc())

lr_by_ticker.show(truncate=False)
lr_by_ticker_pd = lr_by_ticker.toPandas()

# Phân tích accuracy theo ticker - Random Forest
print("\n" + "=" * 100)
print("ACCURACY THEO TICKER - RANDOM FOREST")
print("=" * 100)

rf_by_ticker = rf_predictions.withColumn('correct', col('prediction') == col('label')) \
    .groupBy('ticker') \
    .agg(
        (spark_sum(col('correct').cast('int')) / count('*')).alias('accuracy'),
        count('*').alias('count')
    ) \
    .sort(col('accuracy').desc())

rf_by_ticker.show(truncate=False)
rf_by_ticker_pd = rf_by_ticker.toPandas()

# So sánh 2 model theo ticker
print("\n" + "=" * 100)
print("SO SÁNH 2 MODEL THEO TICKER")
print("=" * 100)

comparison = lr_by_ticker_pd.merge(rf_by_ticker_pd, on='ticker', suffixes=('_LR', '_RF'))
comparison['diff'] = comparison['accuracy_RF'] - comparison['accuracy_LR']
comparison = comparison.sort_values('diff', ascending=False)

print("\n📊 Comparison:")
print(comparison.to_string(index=False))

## PHẦN 13: TRỰC QUAN HÓA KẾT QUẢ

Vẽ các biểu đồ quan trọng:
1. **Accuracy theo ticker**: So sánh hai model
2. **Giá và Moving Averages**: Xu hướng giá thực tế
3. **Predictions vs Actuals**: So sánh dự báo với thực tế
4. **Feature Importance**: Các đặc trưng quan trọng nhất

In [ ]:
# Biểu đồ 1: Accuracy theo ticker
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Plot 1: Accuracy comparison by ticker
comparison_sorted = comparison.sort_values('accuracy_LR', ascending=False)
x = np.arange(len(comparison_sorted))
width = 0.35

axes[0].bar(x - width/2, comparison_sorted['accuracy_LR'], width, label='Logistic Regression', color='steelblue')
axes[0].bar(x + width/2, comparison_sorted['accuracy_RF'], width, label='Random Forest', color='coral')
axes[0].set_xlabel('Ticker', fontsize=11, fontweight='bold')
axes[0].set_ylabel('Accuracy', fontsize=11, fontweight='bold')
axes[0].set_title('Model Accuracy Comparison by Ticker', fontsize=13, fontweight='bold')
axes[0].set_xticks(x)
axes[0].set_xticklabels(comparison_sorted['ticker'])
axes[0].legend()
axes[0].grid(True, alpha=0.3, axis='y')
axes[0].set_ylim([0, 1])

# Plot 2: Feature Importance (Top 10)
feature_importance_df = pd.DataFrame(feature_importance[:10], columns=['Feature', 'Importance'])
axes[1].barh(feature_importance_df['Feature'], feature_importance_df['Importance'], color='green', alpha=0.7)
axes[1].set_xlabel('Importance', fontsize=11, fontweight='bold')
axes[1].set_title('Top 10 Feature Importance (Random Forest)', fontsize=13, fontweight='bold')
axes[1].invert_yaxis()
axes[1].grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.show()

print("✓ Biểu đồ 1 & 2 vẽ xong!")

## PHẦN 14: BACKTEST CHIẾN LƯỢC GIAO DỊCH

**Backtest** là mô phỏng chiến lược giao dịch trên dữ liệu lịch sử để đánh giá hiệu suất thực tế.

### Chiến lược đơn giản:
- **Signal**: Nếu model dự báo `prediction == 1` (giá tăng) → **BUY** (nắm giữ)
- **Signal**: Nếu model dự báo `prediction == 0` (giá giảm) → **SELL** (bán)
- **Return**: Tính lợi nhuận từ sự thay đổi giá

### So sánh:
- **Strategy Return**: Lợi nhuận từ dự báo của model
- **Buy & Hold Return**: Nắm giữ suốt trong test period (baseline)
- **Outperformance**: Chênh lệch giữa strategy và B&H

**Lưu ý**: Backtest này là đơn giản, không tính phí giao dịch, slippage, v.v.

In [ ]:
# Backtest sử dụng Logistic Regression predictions
print("=" * 100)
print("BACKTEST CHIẾN LƯỢC - LOGISTIC REGRESSION")
print("=" * 100)

# Chuyển predictions sang Pandas
lr_pred_pd = lr_predictions.select('time', 'ticker', 'close', 'next_close', 'label', 'prediction').toPandas()
lr_pred_pd['time'] = pd.to_datetime(lr_pred_pd['time'])
lr_pred_pd = lr_pred_pd.sort_values(['ticker', 'time']).reset_index(drop=True)

# Tính Daily Return từ actual prices
lr_pred_pd['actual_return'] = (lr_pred_pd['next_close'] - lr_pred_pd['close']) / lr_pred_pd['close']

# Tính Strategy Return: nếu prediction == 1, buy (nhận return), nếu prediction == 0, không mua (return = 0)
lr_pred_pd['strategy_return'] = lr_pred_pd.apply(
    lambda row: row['actual_return'] if row['prediction'] == 1 else 0, axis=1
)

# Buy & Hold Return (nắm giữ tất cả)
lr_pred_pd['bnh_return'] = lr_pred_pd['actual_return']

# Tính cumulative returns
lr_pred_pd['strategy_cumsum'] = lr_pred_pd.groupby('ticker')['strategy_return'].cumsum() + 1
lr_pred_pd['bnh_cumsum'] = lr_pred_pd.groupby('ticker')['bnh_return'].cumsum() + 1

# Thống kê backtest
print("\n📊 Backtest Results - Logistic Regression:\n")

backtest_summary_lr = []
for ticker in lr_pred_pd['ticker'].unique():
    df_tick = lr_pred_pd[lr_pred_pd['ticker'] == ticker]
    
    strategy_ret = df_tick['strategy_return'].sum() * 100
    bnh_ret = df_tick['bnh_return'].sum() * 100
    win_rate = (df_tick[df_tick['prediction'] == 1]['label'].sum() / 
                (df_tick['prediction'] == 1).sum() * 100) if (df_tick['prediction'] == 1).sum() > 0 else 0
    
    backtest_summary_lr.append({
        'Ticker': ticker,
        'Strategy Return (%)': strategy_ret,
        'B&H Return (%)': bnh_ret,
        'Outperformance (%)': strategy_ret - bnh_ret,
        'Win Rate (%)': win_rate,
        'Trades': (df_tick['prediction'] == 1).sum()
    })

backtest_df_lr = pd.DataFrame(backtest_summary_lr).sort_values('Outperformance (%)', ascending=False)
print(backtest_df_lr.to_string(index=False))

# Tính tổng
print(f"\n{'=' * 100}")
print(f"{'TỔNG':10s} Strategy: {backtest_df_lr['Strategy Return (%)'].sum():7.2f}% | " + 
      f"B&H: {backtest_df_lr['B&H Return (%)'].sum():7.2f}% | " +
      f"Outperformance: {backtest_df_lr['Outperformance (%)'].sum():7.2f}%")

In [ ]:
# Backtest sử dụng Random Forest predictions
print("\n\n" + "=" * 100)
print("BACKTEST CHIẾN LƯỢC - RANDOM FOREST")
print("=" * 100)

# Chuyển predictions sang Pandas
rf_pred_pd = rf_predictions.select('time', 'ticker', 'close', 'next_close', 'label', 'prediction').toPandas()
rf_pred_pd['time'] = pd.to_datetime(rf_pred_pd['time'])
rf_pred_pd = rf_pred_pd.sort_values(['ticker', 'time']).reset_index(drop=True)

# Tính returns
rf_pred_pd['actual_return'] = (rf_pred_pd['next_close'] - rf_pred_pd['close']) / rf_pred_pd['close']
rf_pred_pd['strategy_return'] = rf_pred_pd.apply(
    lambda row: row['actual_return'] if row['prediction'] == 1 else 0, axis=1
)
rf_pred_pd['bnh_return'] = rf_pred_pd['actual_return']

# Thống kê backtest
print("\n📊 Backtest Results - Random Forest:\n")

backtest_summary_rf = []
for ticker in rf_pred_pd['ticker'].unique():
    df_tick = rf_pred_pd[rf_pred_pd['ticker'] == ticker]
    
    strategy_ret = df_tick['strategy_return'].sum() * 100
    bnh_ret = df_tick['bnh_return'].sum() * 100
    win_rate = (df_tick[df_tick['prediction'] == 1]['label'].sum() / 
                (df_tick['prediction'] == 1).sum() * 100) if (df_tick['prediction'] == 1).sum() > 0 else 0
    
    backtest_summary_rf.append({
        'Ticker': ticker,
        'Strategy Return (%)': strategy_ret,
        'B&H Return (%)': bnh_ret,
        'Outperformance (%)': strategy_ret - bnh_ret,
        'Win Rate (%)': win_rate,
        'Trades': (df_tick['prediction'] == 1).sum()
    })

backtest_df_rf = pd.DataFrame(backtest_summary_rf).sort_values('Outperformance (%)', ascending=False)
print(backtest_df_rf.to_string(index=False))

# Tính tổng
print(f"\n{'=' * 100}")
print(f"{'TỔNG':10s} Strategy: {backtest_df_rf['Strategy Return (%)'].sum():7.2f}% | " + 
      f"B&H: {backtest_df_rf['B&H Return (%)'].sum():7.2f}% | " +
      f"Outperformance: {backtest_df_rf['Outperformance (%)'].sum():7.2f}%")

# Vẽ biểu đồ backtest
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Plot 1: Strategy vs B&H - Logistic Regression
tickers_lr = backtest_df_lr['Ticker'].values
x = np.arange(len(tickers_lr))
width = 0.35

axes[0].bar(x - width/2, backtest_df_lr['Strategy Return (%)'], width, label='Strategy', color='green', alpha=0.7)
axes[0].bar(x + width/2, backtest_df_lr['B&H Return (%)'], width, label='Buy & Hold', color='orange', alpha=0.7)
axes[0].axhline(y=0, color='black', linestyle='-', linewidth=0.5)
axes[0].set_xlabel('Ticker', fontsize=11, fontweight='bold')
axes[0].set_ylabel('Return (%)', fontsize=11, fontweight='bold')
axes[0].set_title('Backtest Returns - Logistic Regression', fontsize=13, fontweight='bold')
axes[0].set_xticks(x)
axes[0].set_xticklabels(tickers_lr)
axes[0].legend()
axes[0].grid(True, alpha=0.3, axis='y')

# Plot 2: Strategy vs B&H - Random Forest
tickers_rf = backtest_df_rf['Ticker'].values
x = np.arange(len(tickers_rf))

axes[1].bar(x - width/2, backtest_df_rf['Strategy Return (%)'], width, label='Strategy', color='green', alpha=0.7)
axes[1].bar(x + width/2, backtest_df_rf['B&H Return (%)'], width, label='Buy & Hold', color='orange', alpha=0.7)
axes[1].axhline(y=0, color='black', linestyle='-', linewidth=0.5)
axes[1].set_xlabel('Ticker', fontsize=11, fontweight='bold')
axes[1].set_ylabel('Return (%)', fontsize=11, fontweight='bold')
axes[1].set_title('Backtest Returns - Random Forest', fontsize=13, fontweight='bold')
axes[1].set_xticks(x)
axes[1].set_xticklabels(tickers_rf)
axes[1].legend()
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print("\n✓ Backtest xong!")

## PHẦN 15: KẾT LUẬN VÀ NHẬN XÉT

### Tóm tắt Kết quả

**1. So sánh 2 Mô hình:**
- Logistic Regression: Simple, nhanh, dễ hiểu
- Random Forest: Phức tạp hơn, có thể capture non-linear patterns

**2. Hiệu suất Mô hình:**
- Cả 2 model được đánh giá trên test set (2022-2026)
- Accuracy được tính bằng tỷ lệ dự báo đúng hướng giá

**3. Backtest Trading Strategy:**
- Chiến lược: Mua khi dự báo tăng, bán khi dự báo giảm
- Kết quả: So sánh lợi nhuận với Buy & Hold strategy

### Hạn chế của Nghiên cứu

1. **Không tính phí giao dịch**: Backtest không tính chi phí thực tế (commission, slippage)
2. **Thị trường lý tưởng**: Giả định có thể giao dịch bất cứ lúc nào
3. **Dữ liệu giới hạn**: Chỉ sử dụng OHLCV, không có tin tức hay sự kiện quan trọng
4. **Data leakage**: Có thể có overlapping giữa train/test khi dùng moving averages dài
5. **Không xét rủi ro**: Không tính volatility hoặc maximum drawdown

### Hướng Phát Triển Tiếp Theo

1. **Thêm features nâng cao**:
   - RSI, MACD, Bollinger Bands (technical indicators)
   - Sentiment từ tin tức (NLP)
   - Tương quan với chỉ số thị trường (VN-Index, S&P 500)

2. **Mô hình Machine Learning nâng cao**:
   - Gradient Boosting (XGBoost, LightGBM)
   - Neural Networks (LSTM for time series)
   - Ensemble methods

3. **Quản lý rủi ro**:
   - Stop-loss
   - Position sizing
   - Tính sharpe ratio, sortino ratio

4. **Backtesting thực tế**:
   - Tính phí giao dịch
   - Mô phỏng slippage
   - Kiểm tra kích thước lệnh

### Kết Luận

- **Model nào tốt hơn?** Dựa trên kết quả accuracy và backtest trên test set
- **Có thể áp dụng thực tế?** Cần thêm nhiều cải tiến trước khi live trading
- **Giá trị của ML trong trading?** Có tiềm năng, nhưng không phải magic bullet

**Lưu ý**: Đầu tư chứng khoán có rủi ro cao. Nghiên cứu này chỉ cho mục đích giáo dục, không phải tư vấn đầu tư.

In [ ]:
# Summary Report
print("\n" + "=" * 100)
print("BÁO CÁO TỔNG HỢP - STOCK ANALYSIS & PREDICTION USING PYSPARK")
print("=" * 100)

best_model_name = max(
    [
        ('Logistic Regression', lr_accuracy),
        ('Random Forest', rf_accuracy),
        ('GBTClassifier', gbt_accuracy),
    ],
    key=lambda item: item[1]
)[0]

print(f"""
📊 DATASET THỐNG KÊ:
  • Số cổ phiếu: {len(ticker_list)}
  • Khoảng thời gian: 2013-2026 (13 năm)
  • Tổng records: {df_features.count():,d}
  • Train set: {df_train.count():,d} (2013-2021)
  • Test set: {df_test.count():,d} (2022-2026)

🤖 MÔ HÌNH MACHINE LEARNING:
  • Logistic Regression: {lr_accuracy:.4f} accuracy ({lr_accuracy*100:.2f}%)
  • Random Forest: {rf_accuracy:.4f} accuracy ({rf_accuracy*100:.2f}%)
  • GBTClassifier: {gbt_accuracy:.4f} accuracy ({gbt_accuracy*100:.2f}%)
  • Mô hình tốt nhất: {best_model_name}

📉 SAI SỐ DỰ BÁO:
  • LR Error Rate: {lr_error_rate:.4f} | LogLoss: {lr_logloss:.4f} | Brier: {lr_brier:.4f}
  • RF Error Rate: {rf_error_rate:.4f} | LogLoss: {rf_logloss:.4f} | Brier: {rf_brier:.4f}
  • GBT Error Rate: {gbt_error_rate:.4f} | LogLoss: {gbt_logloss:.4f} | Brier: {gbt_brier:.4f}

📈 FEATURE ENGINEERING:
  • Daily Return, Moving Averages (MA5, MA10)
  • Lag Features (lag1, lag2, lag3)
  • Volatility (5-day std dev)
  • Ticker Encoding

💰 BACKTEST RESULTS (Strategy vs Buy & Hold):
  • LR Total Return: {backtest_df_lr['Strategy Return (%)'].sum():.2f}%
  • RF Total Return: {backtest_df_rf['Strategy Return (%)'].sum():.2f}%
  • LR Outperformance: {backtest_df_lr['Outperformance (%)'].sum():.2f}%
  • RF Outperformance: {backtest_df_rf['Outperformance (%)'].sum():.2f}%

✅ HOÀN THÀNH:
  ✓ Load data từ CSV (27 cổ phiếu)
  ✓ Tiền xử lý dữ liệu
  ✓ Feature Engineering với PySpark Window Functions
  ✓ Train 3 mô hình ML
  ✓ Đánh giá mô hình
  ✓ Phân tích theo ticker
  ✓ Backtest trading strategy
  ✓ Trực quan hóa kết quả

📝 GHI CHÚ:
  • Notebook sử dụng SparkSession local - có thể chạy trên máy cá nhân
  • Code được tối ưu hóa với PySpark Window Functions
  • Phù hợp làm đề tài bảo vệ luận văn tốt nghiệp
  
""")

print("=" * 100)
print("✨ PHÂN TÍCH VÀ DỰ BÁO XU HƯỚNG GIÁ CHỨNG KHOÁN - HOÀN THÀNH ✨")
print("=" * 100)

## PHẦN CUỐI: CÁC CẢI TIẾN ĐỘ CHÍNH XÁC ĐÃ THỰC HIỆN

### Tổng hợp 5 cải tiến:

| # | Cải tiến | Mô tả |
|---|----------|-------|
| 1 | **Tăng ngưỡng label** | `0.2%` → `0.5%` – loại bỏ vùng "nhiễu" quanh 0 |
| 2 | **Thêm features mới** | MA20, MA50, price_vs_ma5/20, Stochastic %K, ATR14, OBV signal |
| 3 | **StandardScaler** | Chuẩn hóa features → giúp Logistic Regression hội tụ tốt hơn |
| 4 | **Class Weighting** | Cân bằng imbalance giữa label 0 và 1 |
| 5 | **Hyperparameter tuning** | RF: numTrees 50→150, maxDepth 10→12; GBT: maxIter 50→150, stepSize 0.1→0.05 |
| 6 | **CrossValidator** | 3-Fold grid search cho Random Forest |

### Lý do từng cải tiến:
- **Ngưỡng label cao hơn**: Phân biệt rõ tăng/giảm, loại mẫu "không rõ xu hướng"
- **MA20/50 + price_vs_ma**: Bắt được xu hướng trung/dài hạn
- **Stochastic %K**: Phát hiện overbought/oversold
- **ATR14**: Đo mức độ biến động thực tế của thị trường
- **OBV signal**: Xác nhận xu hướng qua volume
- **StandardScaler**: Logistic Regression nhạy cảm với scale của features
- **Class Weight**: Tránh model bị bias về phía lớp chiếm đa số


## PHẦN CUỐI: CÁC CẢI TIẾN ĐỘ CHÍNH XÁC ĐÃ THỰC HIỆN

### Tổng hợp 5 cải tiến:

| # | Cải tiến | Mô tả |
|---|----------|-------|
| 1 | **Tăng ngưỡng label** | `0.2%` → `0.5%` – loại bỏ vùng "nhiễu" quanh 0 |
| 2 | **Thêm features mới** | MA20, MA50, price_vs_ma5/20, Stochastic %K, ATR14, OBV signal |
| 3 | **StandardScaler** | Chuẩn hóa features → giúp Logistic Regression hội tụ tốt hơn |
| 4 | **Class Weighting** | Cân bằng imbalance giữa label 0 và 1 |
| 5 | **Hyperparameter tuning** | RF: numTrees 50→150, maxDepth 10→12; GBT: maxIter 50→150, stepSize 0.1→0.05 |
| 6 | **CrossValidator** | 3-Fold grid search cho Random Forest |

### Lý do từng cải tiến:
- **Ngưỡng label cao hơn**: Phân biệt rõ tăng/giảm, loại mẫu "không rõ xu hướng"
- **MA20/50 + price_vs_ma**: Bắt được xu hướng trung/dài hạn
- **Stochastic %K**: Phát hiện overbought/oversold
- **ATR14**: Đo mức độ biến động thực tế của thị trường
- **OBV signal**: Xác nhận xu hướng qua volume
- **StandardScaler**: Logistic Regression nhạy cảm với scale của features
- **Class Weight**: Tránh model bị bias về phía lớp chiếm đa số
